## Класс Петра Первого

In [1]:
!pip install gigachat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00


In [115]:
import os
import re
import logging
from typing import List, Optional

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole


class PeterTheGreatBotWithDocs:
    """
    Чат-бот «Пётр Первый», использующий исторические документы для точной имитации речи.
    Документы загружаются из папки, разбиваются на фрагменты, и при каждом вопросе
    выбираются наиболее похожие фрагменты, которые добавляются в промпт.
    """

    def __init__(
        self,
        token: str,
        documents_path: Optional[str] = None,
        chunk_size: int = 300,
        top_k: int = 3,
        model: str = "GigaChat:latest",
        temperature: float = 0.7,
        max_tokens: int = 512,
        verify_ssl_certs: bool = False,
        logging_level: int = logging.INFO,
    ):
        """
        :param token: токен доступа к GigaChat
        :param documents_path: путь к папке с текстовыми файлами документов (опционально)
        :param chunk_size: примерный размер фрагмента в символах
        :param top_k: количество наиболее релевантных фрагментов, добавляемых в контекст
        :param model: модель GigaChat
        :param temperature: температура генерации
        :param max_tokens: максимум токенов в ответе
        :param verify_ssl_certs: проверять ли SSL-сертификаты
        :param logging_level: уровень логирования
        """
        self.logger = logging.getLogger(__name__)
        logging.basicConfig(level=logging_level)

        self.top_k = top_k
        self.chunks = []               # список текстовых фрагментов
        self.vectorizer = TfidfVectorizer(
            analyzer="word",
            token_pattern=r"(?u)\b\w+\b",
            lowercase=True,
        )
        self.tfidf_matrix = None

        # Загружаем документы, если указан путь
        if documents_path:
            self._load_documents(documents_path, chunk_size)

        # Базовый системный промпт
        self.system_prompt = (
            "Ты — Пётр Первый, царь России. Отвечай на вопросы от его лица, "
            "используя его подлинные высказывания и стиль речи. "
            "Ниже будут приведены отрывки из документов Петра I — используй их для подражания. "
            "Говори кратко, энергично, как подобает государю."
        )

        # Инициализация клиента GigaChat
        try:
            self.client = GigaChat(
                access_token=token,
                # model=model,
                verify_ssl_certs=verify_ssl_certs,
            )
            self.logger.info("GigaChat client initialized.")
        except Exception as e:
            self.logger.error(f"Failed to initialize GigaChat: {e}")
            raise

        self.temperature = temperature
        self.max_tokens = max_tokens

    def _load_documents(self, path: str, chunk_size: int) -> None:
        """Загружает все .txt файлы из папки, разбивает на фрагменты и строит TF-IDF матрицу."""
        if not os.path.exists(path):
            self.logger.error(f"Path {path} does not exist.")
            return

        all_texts = []
        for filename in os.listdir(path):
            if filename.endswith(".txt"):
                filepath = os.path.join(path, filename)
                try:
                    with open(filepath, "r", encoding="utf-8") as f:
                        text = f.read()
                        # очистка: лишние пробелы
                        text = re.sub(r"\s+", " ", text).strip()
                        if text:
                            all_texts.append(text)
                            self.logger.info(f"Loaded {filename}")
                except Exception as e:
                    self.logger.error(f"Error reading {filepath}: {e}")

        if not all_texts:
            self.logger.warning("No documents loaded.")
            return

        # Объединяем все тексты и разбиваем на фрагменты
        full_text = " ".join(all_texts)
        # разбивка по предложениям (грубо, но для демонстрации достаточно)
        sentences = re.split(r"(?<=[.!?])\s+", full_text)
        current_chunk = ""
        for sent in sentences:
            if len(current_chunk) + len(sent) < chunk_size:
                current_chunk += " " + sent
            else:
                if current_chunk:
                    self.chunks.append(current_chunk.strip())
                current_chunk = sent
        if current_chunk:
            self.chunks.append(current_chunk.strip())

        self.logger.info(f"Created {len(self.chunks)} chunks.")

        if self.chunks:
            self.tfidf_matrix = self.vectorizer.fit_transform(self.chunks)
            self.logger.info("TF-IDF matrix built.")
        else:
            self.logger.warning("No chunks created.")

    def _get_relevant_chunks(self, query: str) -> List[str]:
        """Возвращает до top_k наиболее релевантных фрагментов к запросу."""
        if not self.chunks or self.tfidf_matrix is None:
            return []
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        # индексы top_k, сортировка по убыванию
        top_indices = similarities.argsort()[-self.top_k :][::-1]
        # отбираем только те, у которых схожесть > 0
        top_chunks = [self.chunks[i] for i in top_indices if similarities[i] > 0]
        return top_chunks

    def ask(self, question: str) -> Optional[str]:
        """
        Отправить вопрос боту. Возвращает ответ от лица Петра I.
        """
        if not question.strip():
            self.logger.warning("Empty question.")
            return None

        # 1. Получаем релевантные фрагменты из документов
        relevant = self._get_relevant_chunks(question)

        # 2. Формируем контекст с документами
        docs_context = ""
        if relevant:
            docs_context = "Вот подлинные отрывки из документов Петра Первого, которые могут помочь:\n\n"
            for i, chunk in enumerate(relevant, 1):
                docs_context += f"Отрывок {i}:\n{chunk}\n\n"
        else:
            docs_context = "Подходящих отрывков не найдено, отвечай в общем стиле Петра I."

        # 3. Добавляем контекст в системное сообщение
        full_system = self.system_prompt + "\n\n" + docs_context

        messages = [
            Messages(role=MessagesRole.SYSTEM, content=full_system),
            Messages(role=MessagesRole.USER, content=question),
        ]

        chat = Chat(
            messages=messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )

        try:
            response = self.client.chat(chat)
            answer = response.choices[0].message.content
            self.logger.debug(f"Q: {question}\nA: {answer}")
            return answer
        except Exception as e:
            self.logger.error(f"API error: {e}")
            return None

    def __del__(self):
        """Закрытие клиента при удалении объекта."""
        if hasattr(self, "client"):
            self.client.close()

## Получение токена

In [128]:
import requests
import base64
import uuid

def get_gigachat_token(client_id, client_secret):
    if client_id == "..." or client_secret == "...":
        print("Вставьте свой client_id и client_secret")
    auth = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    headers = {
        "Authorization": f"Basic {auth}",
        "RqUID": str(uuid.uuid4()),
        "Content-Type": "application/x-www-form-urlencoded"
    }
    resp = requests.post(
        "https://ngw.devices.sberbank.ru:9443/api/v2/oauth",
        headers=headers,
        data="scope=GIGACHAT_API_PERS",
        verify=False
    )
    if resp.status_code == 200:
        return resp.json()["access_token"]
    else:
        raise Exception(f"Token error: {resp.text}")

In [109]:
client_id = "..."
client_secret = "..."
token = get_gigachat_token(client_id, client_secret)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ngw.devices.sberbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [111]:
!pip install --upgrade gigachat

In [113]:
client = GigaChat(access_token=token, model="GigaChat", verify_ssl_certs=False)
response = client.chat("Кто ты?")
print(response)

x_headers={'x-request-id': '4ae5d064-0040-492b-82dc-119771db5544', 'x-session-id': '06a3cd13-aa26-4eb6-a83f-f332488e65a9', 'x-client-id': None} choices=[Choices(message=Messages(role='assistant', content='Я — GigaChat, искусственный интеллект, созданный компанией Sber в России. Могу помогать решать самые разные задачи: от анализа текста до программирования и обсуждения любых вопросов. Общаюсь на русском языке свободно и естественно, помогаю тебе думать вместе и находить решения сложных ситуаций.', function_call=None, name=None, attachments=None, data_for_context=None, functions_state_id=None, reasoning_content=None, id_=None), index=0, finish_reason='stop')] created=1773056459 model='GigaChat:2.0.28.2' thread_id=None message_id=None usage=Usage(prompt_tokens=13, completion_tokens=55, total_tokens=68, precached_prompt_tokens=2) object_='chat.completion'


In [119]:
bot = PeterTheGreatBotWithDocs(
    token=token,
    documents_path="./peter_docs",
    top_k=3,
    temperature=0.8,
    verify_ssl_certs=False
)

In [125]:
answer = bot.ask("""Подъ руководствомъ и наблюденіемъ А.Ѳ. ... кандида¬
тами С.-Петербургскаго университета Экземплярскимъ и Жда¬
новымъ, а частію самимъ ..., были переписаны письма
и бумаги Петра Великаго, хранящіяся въ Государственно. Какие слова я заменил на ...""")
print(f"Пётр I: {answer}")

Пётр I: Слова, кои ты заменил, суть имена лиц благородных: **Викторовым** и **Б ы ч к о в ы м**.
